# Churn Motive Analysis

Ran after Sentiment Analysis (TF-IDF) to understand based on ABT variables, which category are more likely for a customer to churn.

In [0]:
%pip install shap lightgbm
%pip install shap

In [0]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.lightgbm
import lightgbm as lgb
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

In [0]:
experiment_id = 1426750906214401

mlflow.set_experiment(experiment_id=experiment_id)

In [0]:
query = """
SELECT 
  t1.*, 
  t2.prob_preco,
  t2.prob_concorrente,
  t2.prob_pagamento,
  t2.prob_tecnico
FROM workspace.default.raw_abt2_churn AS t1
LEFT JOIN workspace.default.tbl_prop_feedback AS t2 
    ON t1.customerID = t2.customerID
WHERE t1.TargetChurn = 1
"""

# importing new data
df_motive0 = spark.sql(query).toPandas()
df_motive1 = df_motive0.dropna()


In [0]:
features = [
    # --- 1. Customer Demographics (Numeric Binary) ---
    'isMale',
    'SeniorCitizen',
    'isPartner',
    'HasDependents',
    'MonthlyIncome',

    # --- 2. Account & Tenure ---
    'nbrTenure',                  # Continuous tenure
    'isNewCustomer',              # High-risk early churn flag (tenure <= 6)
    'isPaperlessBilling',
    'isContractOneYear',          # Baseline = Month-to-month
    'isContractTwoYear',

    # --- 3. Payment Methods (One flag dropped as reference category) ---
    'isPaymentMethod_ElectronicCheck',  # Known highest-risk payment method
    'isPaymentMethod_MailedCheck',
    'isPaymentMethod_BankTransfer',
    'isPaymentMethod_CreditCard',

    # --- 4. Core Telecom Services ---
    'hasPhoneService',
    'hasMultipleLines',
    'hasInternetService',
    'isInternetService_DSL',
    'isInternetService_Fiber',          # Known high churn driver in EDA

    # --- 5. Add-on Internet Services ---
    'hasOnlineSecurity',
    'hasOnlineBackup',
    'hasDeviceProtection',
    'hasTechSupport',
    'hasStreamingTV',
    'hasStreamingMovies',
    'nbrInternetServicesAddOnQty',      # Total count of add-ons
    'hasSecurityBundle',                # High-loyalty flag

    # --- 6. High-Stakes Business Interactions (The Revenue Levers) ---
    'isM2M_dual',                       # The #1 revenue hemorrhage segment (47.3% churn)
    'isM2M_internetOnly',
    'isOneYear_dual',
    'isTwoYear_dual',

    # --- 7. Billing & Financial Ratios ---
    'MonthlyCharges',
    'TotalCharges',                     # Imputed by your champion Ridge model
    'TotalSpend',                       # Interaction (tenure * MonthlyCharges)
    'chargeRatio',                      # MonthlyCharges / MonthlyIncome (affordability stress)
    'numServices',                      # Service breadth
    'chargesPerService',                # Value perception

    # --- 8. Sentiment Analysis ---
    'prob_preco',
    'prob_concorrente',
    'prob_pagamento',
    'prob_tecnico',
]


In [0]:
df_motive2 = df_motive1[features]

def train_and_log_motives(X, targets, prob_cols):
    results = {}
    
    # Start a parent run so all 4 motives are grouped together
    with mlflow.start_run(run_name="Motive_SHAP_Analysis") as parent_run:
        
        for motive in prob_cols:
            # Start a nested run for each motive
            with mlflow.start_run(run_name=motive, nested=True):
                print(f"Training {motive}...")
                y = targets[motive].values
                
                X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
                
                model = lgb.LGBMRegressor(
                    n_estimators=500, num_leaves=31, learning_rate=0.05,
                    min_child_samples=20, reg_alpha=0.1, reg_lambda=0.1,
                    min_split_gain=0.005, subsample=0.8, colsample_bytree=0.8,
                    random_state=42, verbose=-1
                )
                model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)],
                          callbacks=[lgb.early_stopping(50, verbose=False)])
                
                # Metrics
                pred = model.predict(X_te)
                r2 = r2_score(y_te, pred)
                mae = mean_absolute_error(y_te, pred)
                
                # Log metrics and model to MLflow
                mlflow.log_metric("test_r2", r2)
                mlflow.log_metric("test_mae", mae)
                
                # Log the model so we can reuse it for scoring later
                mlflow.lightgbm.log_model(model, artifact_path="model")
                
                # SHAP
                explainer = shap.TreeExplainer(model)
                shap_vals = explainer.shap_values(X_te)
                
                imp = pd.DataFrame({
                    "feature": X_te.columns,
                    "mean_abs_shap": np.abs(shap_vals).mean(axis=0)
                }).sort_values("mean_abs_shap", ascending=False)
                
                # Save the importance table as an artifact in MLflow
                imp.to_csv(f"{motive}_feature_importance.csv", index=False)
                mlflow.log_artifact(f"{motive}_feature_importance.csv")
                
                results[motive] = {"model": model, "importance": imp, "r2": r2}
                
    return results

In [0]:
prob_cols = ["prob_preco", "prob_concorrente", "prob_pagamento", "prob_tecnico"]

# 1. Create targets directly from the original dataframe
targets = df_motive2[prob_cols].copy()

# 2. Create X by dropping the non-features
cols_to_drop = [c for c in ["customerID", "TargetChurn"] + prob_cols if c in df_motive2.columns]
X = df_motive2.drop(columns=cols_to_drop).copy()

# 3. Force categories
cat_cols = ["ContractTier", "tenureGroup", "ServiceTier"]
for c in cat_cols:
    if c in X.columns:
        X[c] = X[c].astype("category")

print(f"X shape: {X.shape}")
print(f"targets shape: {targets.shape}")
# This should print (1405, 37) and (1405, 4)

In [0]:
print("DataFrame shape:", df_motive2.shape)
print("Columns:", df_motive2.columns.tolist())

# Running Model


In [0]:
import mlflow
import mlflow.lightgbm
import lightgbm as lgb
import shap
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

prob_cols = ["prob_preco", "prob_concorrente", "prob_pagamento", "prob_tecnico"]

df_fresh = df_motive2.copy()

# 2. Define X and targets
# Drop customerID, TargetChurn, and the prob columns to create X
cols_to_drop = [c for c in ["customerID", "TargetChurn"] + prob_cols if c in df_fresh.columns]
X = df_fresh.drop(columns=cols_to_drop).copy()
targets = df_fresh[prob_cols].copy()

# Force categories for LightGBM
cat_cols = ["ContractTier", "tenureGroup", "ServiceTier"]
for c in cat_cols:
    if c in X.columns:
        X[c] = X[c].astype("category")

print(f"Data ready. X shape: {X.shape}, targets shape: {targets.shape}\n")

# 3. Training & MLflow Function
def train_and_log_motives(X, targets, prob_cols):
    results = {}
    
    # Parent run groups all 4 motives together
    with mlflow.start_run(run_name="Motive_SHAP_Training") as parent_run:
        
        for motive in prob_cols:
            # Nested run for each motive
            with mlflow.start_run(run_name=motive, nested=True):
                print(f"Training {motive}...")
                y = targets[motive].values
                
                X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
                
                model = lgb.LGBMRegressor(
                    n_estimators=500, num_leaves=31, learning_rate=0.05,
                    min_child_samples=20, reg_alpha=0.1, reg_lambda=0.1,
                    min_split_gain=0.005, subsample=0.8, colsample_bytree=0.8,
                    random_state=42, verbose=-1
                )
                
                model.fit(
                    X_tr, y_tr, 
                    eval_set=[(X_te, y_te)],
                    callbacks=[lgb.early_stopping(50, verbose=False)]
                )
                
                # Metrics
                pred = model.predict(X_te)
                r2 = r2_score(y_te, pred)
                mae = mean_absolute_error(y_te, pred)
                
                # Log to MLflow
                mlflow.log_metric("test_r2", r2)
                mlflow.log_metric("test_mae", mae)
                mlflow.lightgbm.log_model(model, artifact_path="model")
                
                # SHAP
                explainer = shap.TreeExplainer(model)
                shap_vals = explainer.shap_values(X_te)
                
                imp = pd.DataFrame({
                    "feature": X_te.columns,
                    "mean_abs_shap": np.abs(shap_vals).mean(axis=0)
                }).sort_values("mean_abs_shap", ascending=False)
                
                # Save and log importance table
                file_name = f"{motive}_feature_importance.csv"
                imp.to_csv(file_name, index=False)
                mlflow.log_artifact(file_name)
                
                # Save results for later use
                results[motive] = {
                    "model": model, 
                    "importance": imp, 
                    "r2": r2,
                    "run_id": mlflow.active_run().info.run_id
                }
                print(f"  -> {motive} done! R2: {r2:.3f}")
                
    return results

# 4. RUN IT
print("Starting MLflow Experiment...\n")
motive_results = train_and_log_motives(X, targets, prob_cols)
print("\nAll done! Check the MLflow Experiments tab in Databricks.")